In [22]:
import numpy as np
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.data.dataset import ReactionDatasetService
from prophet_gp.features.gauche_adapter import GaucheFeaturizerRegistry, _morgan_fp_featuriser


In [24]:
featurizers = GaucheFeaturizerRegistry()
featuriser = "morgan_fp"
reactant_smiles = ["Nc1cccc2ccccc12", "Nc1ccc2ccccc2c1", "Nc1cccc2c(N)cccc12", "Nc1cccc2cccc(N)c12", "Nc1cc2ccccc2cc1N", "Nc1ccc2cc(N)ccc2c1"] # ["C1=CC=CC=C1", "C1=CC=CC=C1"]
#reactant_smiles = ["Nc1cccc2c(N)cccc12", "Nc1cccc2cccc(N)c12", "Nc1cc2ccccc2cc1N", "Nc1ccc2cc(N)ccc2c1"] # ["C1=CC=CC=C1", "C1=CC=CC=C1"]

per_molecule = _morgan_fp_featuriser(reactant_smiles, radius=1, n_bits=2048)

# 모든 행에서 값이 동일한 열(상수 열) 제거; 행이 1개면 비교 불가로 전열 유지
n_rows = per_molecule.shape[0]

if n_rows <= 1:
    per_molecule_filtered = per_molecule.copy()
    kept_column_indices = np.arange(per_molecule.shape[1], dtype=np.int64)
    removed_column_indices = np.array([], dtype=np.int64)
else:
    is_varying = np.ptp(per_molecule, axis=0) != 0
    kept_column_indices = np.flatnonzero(is_varying)
    removed_column_indices = np.flatnonzero(~is_varying)
    per_molecule_filtered = per_molecule[:, kept_column_indices]

print("원본 shape:", per_molecule.shape)
print("제거된 열 개수 (모든 행 동일):", removed_column_indices.size)
print("유지된 원본 column indices:", kept_column_indices)
print("filtered shape:", per_molecule_filtered.shape)
print("filtered array:\n", per_molecule_filtered)



원본 shape: (6, 2048)
제거된 열 개수 (모든 행 동일): 2041
유지된 원본 column indices: [ 875  888  910 1088 1357 1855 1984]
filtered shape: (6, 7)
filtered array:
 [[0. 0. 1. 1. 1. 1. 0.]
 [1. 1. 0. 1. 0. 1. 0.]
 [0. 0. 1. 1. 1. 0. 0.]
 [0. 0. 1. 1. 0. 1. 1.]
 [1. 0. 1. 1. 0. 1. 0.]
 [1. 1. 0. 0. 0. 1. 0.]]


In [15]:
featurizers = GaucheFeaturizerRegistry()
featuriser = "morgan_fp"
reactant_smiles = ["c1ccc2c(c1)c(N)ccc2", "Nc1ccc2ccccc2c1"] # ["C1=CC=CC=C1", "C1=CC=CC=C1"]

per_molecule = featurizers.featurize(
    reactant_smiles, name=featuriser
)

molecule_1 = per_molecule[0]
molecule_2 = per_molecule[1]

different_indices = np.where(molecule_1 != molecule_2)[0]
print(different_indices)

[  90  203  322  397  494  598  647  875  888  910 1039 1104 1168 1357
 1573 1878]
